Use TF-IDF or Bag of Words to vectorize review text by experimenting with n-grams, stopword removal, and stemming/lemmatization, then optimizing feature selection using chi-square or mutual information scoring.

In [ ]:

import nltk
import os
import shutil

# Remove the corrupted WordNet download, if present
for path in nltk.data.path:
    wordnet_zip = os.path.join(path, "corpora", "wordnet.zip")
    wordnet_folder = os.path.join(path, "corpora", "wordnet")

    if os.path.exists(wordnet_zip):
        os.remove(wordnet_zip)

    if os.path.isdir(wordnet_folder):
        shutil.rmtree(wordnet_folder)

# Download WordNet again
nltk.download("wordnet", force=True)
nltk.download("omw-1.4", force=True)

print("WordNet download completed.")


In [ ]:
import re
import nltk
from nltk.stem import SnowballStemmer, WordNetLemmatizer
from nltk.corpus import wordnet

from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer, ENGLISH_STOP_WORDS
from sklearn.feature_selection import SelectKBest, chi2
from sklearn.metrics import accuracy_score, f1_score
from sklearn.linear_model import LogisticRegression

In [ ]:
task19_df = task18_df.copy()

print("Dataset shape:", task19_df.shape)
print("Required columns available:", {"review_text", "rating"}.issubset(task19_df.columns))

# Remove rows with missing review text or rating
task19_df = task19_df.dropna(subset=['review_text', 'rating']).copy()

task19_df["review_text"] = task19_df["review_text"].astype(str)

# Convert numerical ratings into sentiment classes
def rating_to_sentiment(rating):
    if rating >= 4:
        return "Positive"
    elif rating <= 2:
        return "Negative"
    else:
        return "Neutral"

task19_df['sentiment_target'] = (task19_df['rating'].apply(rating_to_sentiment))
print("\nSentiment target distribution:")
print(task19_df["sentiment_target"].value_counts())

print("\nPrepared dataset shape:", task19_df.shape)

print(task19_df.columns.to_list())


In [ ]:
stemmer = SnowballStemmer('english')
lemmatizer = WordNetLemmatizer()

Experiment with stemming and lemmatization

In [ ]:
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"[^a-z\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def stem_text(text):
    words = clean_text(text).split()
    return " ".join(stemmer.stem(word) for word in words)

def lemmatize_text(text):
    words = clean_text(text).split()
    return " ".join(lemmatizer.lemmatize(word) for word in words)


task19_df["text_clean"] = task19_df["review_text"].apply(clean_text)
task19_df["text_stemmed"] = task19_df["review_text"].apply(stem_text)
task19_df["text_lemmatized"] = task19_df["review_text"].apply(lemmatize_text)

print(task19_df[["review_text", "text_clean", "text_stemmed", "text_lemmatized"]].head(5).to_string(index=False))

Split the data before feature selection

In [ ]:
x_train, x_test, y_train, y_test = train_test_split(task19_df[['text_clean', 'text_stemmed', 'text_lemmatized']], task19_df["sentiment_target"], test_size=0.20, random_state=42, stratify=task19_df['sentiment_target'])

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

Compare Bag of Words and TF-IDF

In [ ]:
results = []
best_model = None
best_vectorizer = None
best_selector = None
best_config = None
best_f1 = -1

text_cols = {
    "Original": "text_clean",
    "Stemming": "text_stemmed",
    "Lemmatization": "text_lemmatized"
}

for preprocessing, text_col in text_cols.items():
    for vectorizer_name in ['BoW', 'TF-IDF']:
        for ngram_range in [(1, 1), (1, 2)]:
            for remove_stopwords in [False, True]:

                vectorizer_class = (CountVectorizer if vectorizer_name=='BoW' else TfidfVectorizer)

                vectorizer = vectorizer_class(max_features=3000, ngram_range=ngram_range, stop_words='english' if remove_stopwords else None, min_df=2)

                Xtr = vectorizer.fit_transform(x_train[text_col])
                Xte = vectorizer.transform(x_test[text_col])

                # Chi-square feature selection
                number_to_select = min(1000, Xtr.shape[1])

                selector = SelectKBest(score_func=chi2, k=number_to_select)

                Xtr_selected = selector.fit_transform(Xtr, y_train)
                Xte_selected = selector.transform(Xte)

                model = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42)

                model.fit(Xtr_selected, y_train)
                predictions = model.predict(Xte_selected)

                accuracy = accuracy_score(y_test, predictions)
                macro_f1 = f1_score(y_test, predictions, average='macro')


                config = {
                    "Preprocessing": preprocessing,
                    "Vectorizer": vectorizer_name,
                    "N-grams": str(ngram_range),
                    "Stopword Removal": remove_stopwords,
                    "Features Before Selection": Xtr.shape[1],
                    "Features Selected": number_to_select,
                    "Accuracy": accuracy,
                    "Macro F1": macro_f1
                }
                results.append(config)

                if macro_f1 > best_f1:
                    best_f1 = macro_f1
                    best_model = model
                    best_vectorizer = vectorizer
                    best_selector = selector
                    best_config = config.copy()

result_df = pd.DataFrame(results).sort_values(by='Macro F1', ascending=False).reset_index(drop=True)

print("Top 10 configurations:")
print(results_df.head(10).round(4).to_string(index=False))

print("\nBest configuration:")
print(best_config)

Inspect the selected features

In [ ]:
feature_names = best_vectorizer.get_feature_names_out()
chi2_scores = best_selector.scores_
selected_mask = best_selector.get_support()

selected_features = pd.DataFrame({
    "Feature": feature_names[selected_mask],
    "Chi_Square_Score": chi2_scores[selected_mask]
}).sort_values("Chi_Square_Score", ascending=False).reset_index(drop=True)

print("Top 20 selected text features:")
print(selected_features.head(20).round(3).to_string(index=False))

Save the results

In [ ]:

# Save the comparison table and selected feature list
results_df.to_csv("task19_vectorization_comparison.csv", index=False)
selected_features.to_csv("task19_selected_features.csv", index=False)

print("Saved:")
print("- task19_vectorization_comparison.csv")
print("- task19_selected_features.csv")

print("\nBest Macro F1:", round(best_f1, 4))
print("Selected feature count:", len(selected_features))
